In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR

In [2]:
sim = SIMULATOR()

# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/fir_filter_2_col/'
version=""

sim.compileHexToAsm(kernel_path, version)

Hex to ASM
Processing file: ./kernels/fir_filter_2_col/instructions_hex.csv...
Creating file: ./kernels/fir_filter_2_col/instructions_asm.csv


In [3]:
# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_number = 1 
column_usage = [True, True] 
nInstrPerCol = 20 
imem_add_start = 0 
srf_spm_addres = 0 

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [4]:
# --------------------------------------------
#                DATA SIZES
# --------------------------------------------
# DISCO-CGRA Configuration
nRCs = 4
nElementsPerVWRSlice = 32
nVWR = 128
nColsCGRA = 2


In [14]:
# Our test
N_COEFS = 11
N_DATA = 512

data = np.array([i for i in range(N_DATA)])
coeffs = np.array([i for i in range(N_COEFS)])
output = np.zeros((N_DATA), dtype=int)

print(data)
print(coeffs)

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

In [6]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [17]:
# --------------------------------------------
#                LOAD SPM DATA
# --------------------------------------------
# SPM[0] = SRF
# SPM[1] = Coefs
# SPM[2->5] = Data 
# SPM[6->9] = Out
# --------------------------------------------
# SRF[0] = SPM Coefs 
# SRF[1] = SPM Data
# SRF[2] = SPM Output
# SRF[3] = 32 - #Coefs
# SRF[4] = #data
# SRF[5] = #coefs
# SRF[6] = -
# SRF[7] = -
# --------------------------------------------

# Default SPM lines
srf_spm_line = 0
coeffs_spm_line = 1
data_spm_line = 2
data_2_spm_line = 4
out_spm_line = 6
out_2_spm_line = 10

# Default SRF values
srf = [0 for i in range(N_ELEMS_PER_VWR)]
# Depend on the dimensions

# Col 0
srf[0]  = coeffs_spm_line
srf[1]  = data_spm_line
srf[2]  = out_spm_line
srf[3]  = 32 - N_COEFS
srf[4]  = int(N_DATA/2)
srf[5]  = N_COEFS 
srf[6]  = 0      # Not used
srf[7]  = 0      # Not used
# Col 1
srf[8]  = coeffs_spm_line
srf[9]  = data_2_spm_line
srf[10] = out_2_spm_line
srf[11] = 32 - N_COEFS
srf[12] = int(N_DATA/2)
srf[13] = N_COEFS 
srf[14] = 0      # Not used 
srf[15] = 0      # Not used
sim.setSPMLine(srf_spm_line, srf.copy())

# Coefs to SPM in vectors of 128 elements
vec_coefs = np.concatenate((coeffs.copy(), np.zeros(nVWR - len(coeffs), dtype=int)))[:128]
sim.setSPMLine(coeffs_spm_line, vec_coefs)
# Data to SPM in vectors of 128 elements
for i in range(int(N_DATA/nVWR)):
    data_vector = data[i*nVWR:(i+1)*nVWR]
    sim.setSPMLine(data_spm_line + i, data_vector.copy())
# Output to SPM (zeroed) in vectors of 128 elements
for i in range(int(N_DATA/nVWR)):
    out_vector = output[i*nVWR:(i+1)*nVWR]
    sim.setSPMLine(out_spm_line + i, out_vector.copy())


In [18]:
sim.displaySPMLine(0)
sim.displaySPMLine(1)
sim.displaySPMLine(2)
sim.displaySPMLine(3)
sim.displaySPMLine(4)

SPM 0: [1, 2, 6, 21, 256, 11, 0, 0, 1, 4, 10, 21, 256, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]
SPM 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]
SPM 2: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 

In [8]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

ASM to Hex
Processing file: ./kernels/mmul/instructions_asm_bb16x16_1col_param.csv...
Creating file: ./kernels/mmul/dsip_bitstream__bb16x16_1col_param.h
Creating file: ./kernels/mmul/instructions_hex_bb16x16_1col_param_autogen.csv


Finally, we load the kernel into the internal memory of the specialized units and run it.

In [19]:
# --------------------------------------------
#                 LOAD KERNEL
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version, kernel_number=kernel_number)

# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

sim.run(kernel_number, display_ops=display_ops, max_iter=3000)

Processing file: ./kernels/fir_filter_2_col/instructions_hex.csv...
---------------------
     PC[0]: 0
---------------------
LSU: LOR R0, SRF(0), ZERO/LD.VWR SRF --> ALU res = 1
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: NOP (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 0
LCU: LOR R0, ZERO, ZERO --> ALU res = 0
---------------------
     PC[1]: 0
---------------------
LSU: NOP/NOP --> ALU res = 0
RC0: LOR VWR_C, R1, RCT, ZERO --> ALU res = 0
RC1: SADD VWR_C, VWR_C, RCT --> ALU res = 0
RC2: SADD VWR_C, VWR_C, RCT --> ALU res = 0
RC3: SADD , VWR_C, RCT --> ALU res = 0
MXCU: NOP (VWR selected: 2, not writting SRF, R0: 0) --> ALU res = 0
LCU: NOP --> ALU res = 0
---------------------
     PC[0]: 1
---------------------
LSU: LOR R1, SRF(1), ZERO/NOP --> ALU res = 2
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: NOP (VWR selected: 0, not writting SRF, R0: 0) -

IndexError: index 64 is out of bounds for axis 0 with size 64

We can check it more rigorously. We can define our function in python and check that the output matches the CGRA output.

In [10]:
def mmul (in_A, in_B, nRowsA, nColsA, nColsB):
    out = np.zeros(nRowsA*nColsB)
    for i in range(nRowsA):
        for j in range(nColsB):
            sum = 0
            for k in range(nColsA):
                sum += int(in_A[i*nColsA + k] * in_B[k*nColsB + j])
            out[i*nColsB + j] = sum
    return [int(elem) for elem in out]

In [11]:
from itertools import groupby

def comprimir_rangos(arr):
    arr.sort()  # Asegurarse de que esté ordenado
    rangos = []
    
    for _, grupo in groupby(enumerate(arr), lambda x: x[1] - x[0]):
        grupo = [x[1] for x in grupo]  # Extraer los valores
        if len(grupo) > 1:
            rangos.append(f"{grupo[0]}-{grupo[-1]}")
        else:
            rangos.append(f"{grupo[0]}")

    return ", ".join(rangos)

def imprimir_por_linea(arr, tam_linea=8):
    for i in range(0, len(arr), tam_linea):
        print([int(x) for x in arr[i:i+tam_linea]])

In [12]:
def reordenarC(disco_cgra_res):
    nBloques = 16
    tamBloque = 8
    # Reorganizar los bloques en el orden correcto
    array_ordenado = []
    for i in range(nColsCGRA):
        ini = 2*tamBloque*i
        for j in range (nRCs):
            array_ordenado.extend(disco_cgra_res[ini:ini+2*tamBloque])
            ini += 4*tamBloque
    return array_ordenado

In [13]:
# Get output from the CGRA
disco_cgra_res_0 = sim.getSPMLine(c_spm_line)
disco_cgra_res_1 = sim.getSPMLine(c_spm_line + 1)
print(disco_cgra_res_0)
print(disco_cgra_res_1)

out_0 = reordenarC(disco_cgra_res_0)
out_1 = reordenarC(disco_cgra_res_1)

# Unir los bloques de C adecuadamente
disco_cgra_res = []
for i in range(0,N_ELEMS_PER_VWR, 8):
    disco_cgra_res.extend(out_0[i:i+8])
    disco_cgra_res.extend(out_1[i:i+8])

[3264, 53220, 54021, 54822, 52215, 3408, 56424, 53745, 3480, 58026, 55275, 8448, 8565, 8682, 8799, 0, 8916, 9033, 0, 9150, 9267, 0, 44736, 45420, 46104, 46788, 0, 47472, 48156, 0, 48840, 49524, 13632, 74118, 75243, 76368, 63069, 14424, 78618, 64923, 14820, 80868, 66777, 18816, 19095, 19374, 19653, 0, 19932, 20211, 0, 20490, 20769, 0, 55104, 55950, 56796, 57642, 0, 58488, 59334, 0, 60180, 61026, 24000, 95016, 96465, 97914, 73923, 25440, 100812, 76101, 26160, 103710, 78279, 29184, 29625, 30066, 30507, 0, 30948, 31389, 0, 31830, 32271, 0, 65472, 66480, 67488, 68496, 0, 69504, 70512, 0, 71520, 72528, 34368, 115914, 117687, 119460, 84777, 36456, 123006, 87279, 37500, 126552, 89781, 39552, 40155, 40758, 41361, 0, 41964, 42567, 0, 43170, 43773, 0, 75840, 77010, 78180, 79350, 0, 80520, 81690, 0, 82860, 84030]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [14]:
errors_idx = []
expected_output = mmul(matrix_A, matrix_B, nRowsA, nColsA, nColsB)
for i in range(len(expected_output)):
    if expected_output[i] != disco_cgra_res[i]:
        errors_idx.append(i)
if len(errors_idx) == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(len(errors_idx)) + " errors.")
    print(comprimir_rangos(errors_idx))
    print("DISCO-CGRA result:")
    imprimir_por_linea(disco_cgra_res)
    print("Expected result:")
    imprimir_por_linea(expected_output)

Oops, something went wrong. There are 252 errors.
1-31, 33-63, 65-95, 97-255
DISCO-CGRA result:
[3264, 53220, 54021, 54822, 52215, 3408, 56424, 53745]
[0, 0, 0, 0, 0, 0, 0, 0]
[3480, 58026, 55275, 8448, 8565, 8682, 8799, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[13632, 74118, 75243, 76368, 63069, 14424, 78618, 64923]
[0, 0, 0, 0, 0, 0, 0, 0]
[14820, 80868, 66777, 18816, 19095, 19374, 19653, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[24000, 95016, 96465, 97914, 73923, 25440, 100812, 76101]
[0, 0, 0, 0, 0, 0, 0, 0]
[26160, 103710, 78279, 29184, 29625, 30066, 30507, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[34368, 115914, 117687, 119460, 84777, 36456, 123006, 87279]
[0, 0, 0, 0, 0, 0, 0, 0]
[37500, 126552, 89781, 39552, 40155, 40758, 41361, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[8916, 9033, 0, 9150, 9267, 0, 44736, 45420]
[0, 0, 0, 0, 0, 0, 0, 0]
[46104, 46788, 0, 47472, 48156, 0, 48840, 49524]
[0, 0, 0, 0, 0, 0, 0, 0]
[19932, 20211, 0, 20490, 20769, 0, 55104, 55950]
[0, 0, 0, 0, 0, 0, 0, 0]
[56796, 57642, 0, 58488, 59334, 0, 60180, 6102